In [2]:
import os
import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

POSTGRES_USER = "postgres"
POSTGRES_PASSWORD = "madhu"  # Aapka password
HOST = "127.0.0.1"
PORT = "5432"
DB_NAME = "insight360_db"
SCHEMA_NAME = "insight360"

print("Step 1: Connecting to PostgreSQL server...")

# Default 'postgres' DB se connect karke database recreate karenge
conn = psycopg2.connect(
    dbname="postgres", user=POSTGRES_USER, password=POSTGRES_PASSWORD, host=HOST, port=PORT
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

cur.execute(f"DROP DATABASE IF EXISTS {DB_NAME};")
cur.execute(f"CREATE DATABASE {DB_NAME};")
cur.close()
conn.close()

print(f"✅ Database '{DB_NAME}' created cleanly!")

# Target Database connect karke schema aur tables banayenge
conn = psycopg2.connect(
    dbname=DB_NAME, user=POSTGRES_USER, password=POSTGRES_PASSWORD, host=HOST, port=PORT
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

cur.execute(f"CREATE SCHEMA {SCHEMA_NAME};")

LOAD_PLAN = [
    "dim_date.csv", "dim_store.csv", "dim_product.csv", "dim_customer.csv",
    "fact_sales.csv", "fact_returns.csv", "fact_inventory_snapshot.csv", "fact_staffing.csv"
]

def get_sql_type(dtype, col_name):
    if any(k in col_name for k in ["key", "id", "code", "zip"]): return "TEXT"
    if "int" in str(dtype): return "BIGINT"
    if "float" in str(dtype): return "NUMERIC"
    if "date" in col_name: return "DATE"
    return "TEXT"

print("\nStep 2: Auto-generating tables & loading data...")
for csv_file in LOAD_PLAN:
    table_name = csv_file.replace(".csv", "")
    if not os.path.exists(csv_file):
        print(f"❌ File not found: {csv_file}")
        continue

    df_sample = pd.read_csv(csv_file, nrows=100)
    col_defs = [f'"{col}" {get_sql_type(df_sample[col].dtype, col.lower())}' for col in df_sample.columns]

    cur.execute(f'CREATE TABLE {SCHEMA_NAME}."{table_name}" ({", ".join(col_defs)});')
    with open(csv_file, "r", encoding="utf-8") as f:
        cur.copy_expert(f'COPY {SCHEMA_NAME}."{table_name}" FROM STDIN WITH (FORMAT csv, HEADER true, NULL \'\')', f)
    
    cur.execute(f'SELECT COUNT(*) FROM {SCHEMA_NAME}."{table_name}";')
    count = cur.fetchone()[0]
    print(f"✅ Loaded {table_name:<25} | DB Rows: {count:,}")

cur.close()
conn.close()
print("\n🎉 ALL CSVs SUCCESSFULLY INGESTED INTO POSTGRESQL!")

Step 1: Connecting to PostgreSQL server...
✅ Database 'insight360_db' created cleanly!

Step 2: Auto-generating tables & loading data...
✅ Loaded dim_date                  | DB Rows: 730
✅ Loaded dim_store                 | DB Rows: 216
✅ Loaded dim_product               | DB Rows: 4,200
✅ Loaded dim_customer              | DB Rows: 850,000
✅ Loaded fact_sales                | DB Rows: 4,200,000
✅ Loaded fact_returns              | DB Rows: 220,000
✅ Loaded fact_inventory_snapshot   | DB Rows: 968,188
✅ Loaded fact_staffing             | DB Rows: 11,128

🎉 ALL CSVs SUCCESSFULLY INGESTED INTO POSTGRESQL!


In [3]:
import psycopg2

POSTGRES_USER = "postgres"
POSTGRES_PASSWORD = "madhu"  # <-- Replace with your exact password
HOST = "127.0.0.1"
PORT = "5432"
DB_NAME = "insight360_db"

print("Step 1: Resetting schema structure...")
conn = psycopg2.connect(
    dbname=DB_NAME, user=POSTGRES_USER, password=POSTGRES_PASSWORD, host=HOST, port=PORT
)
conn.autocommit = True
cursor = conn.cursor()

# Drop existing schema to clear mismatched column structures
cursor.execute("DROP SCHEMA IF EXISTS insight360 CASCADE;")
cursor.execute("CREATE SCHEMA insight360;")

# Re-read and apply schema_ddl.sql
with open("schema_ddl.sql", "r", encoding="utf-8") as f:
    sql_script = f.read()

cursor.execute(sql_script)
print("✅ Database schema cleanly reset and recreated!")

cursor.close()
conn.close()

Step 1: Resetting schema structure...
✅ Database schema cleanly reset and recreated!


In [7]:
%run load_data.py

2026-08-06 12:42:27 | INFO     | ======================================================================
2026-08-06 12:42:27 | INFO     | Insight360 Phase 4 — Data Load Starting
2026-08-06 12:42:27 | INFO     | Data directory: C:\Users\madhu\Downloads\Insight360-BI-Platform
2026-08-06 12:42:27 | INFO     | Target schema : insight360
2026-08-06 12:42:27 | INFO     | ======================================================================
2026-08-06 12:42:27 | INFO     | Connected to database 'insight360_db' on 127.0.0.1:5432 as user 'postgres'
2026-08-06 12:42:27 | INFO     | ----------------------------------------------------------------------
2026-08-06 12:42:27 | INFO     | Loading dim_date.csv -> insight360.dim_date
2026-08-06 12:42:27 | INFO     |   Source CSV row count: 730
2026-08-06 12:42:27 | INFO     |   Truncated insight360.dim_date before load
2026-08-06 12:42:27 | ERROR    |   ERROR loading dim_date: column "fiscal_year" of relation "dim_date" does not exist

2026-08-06 12:42

SystemExit: 1

In [8]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# Database credentials
DB_CONFIG = {
    "host": "127.0.0.1",
    "port": "5432",
    "dbname": "insight360_db",
    "user": "postgres",
    "password": "madhu",  # <-- UPDATE YOUR PASSWORD HERE
}

SCHEMA_NAME = "insight360"

LOAD_PLAN = [
    "dim_date.csv",
    "dim_store.csv",
    "dim_product.csv",
    "dim_customer.csv",
    "fact_sales.csv",
    "fact_returns.csv",
    "fact_inventory_snapshot.csv",
    "fact_staffing.csv",
]

print("Step 1: Connecting to PostgreSQL...")
conn = psycopg2.connect(**DB_CONFIG)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

# Reset Schema
print(f"Step 2: Resetting schema '{SCHEMA_NAME}'...")
cur.execute(f"DROP SCHEMA IF EXISTS {SCHEMA_NAME} CASCADE;")
cur.execute(f"CREATE SCHEMA {SCHEMA_NAME};")

# Map pandas dtypes to SQL data types
def get_sql_type(dtype, col_name):
    if "key" in col_name or "id" in col_name or "code" in col_name or "zip" in col_name:
        return "TEXT"
    if "int" in str(dtype):
        return "BIGINT"
    if "float" in str(dtype):
        return "NUMERIC"
    if "date" in col_name:
        return "DATE"
    return "TEXT"

print("\nStep 3: Auto-generating tables from CSV headers & loading data...")
conn.set_isolation_level(0) # Normal transaction mode
conn.autocommit = False

for csv_file in LOAD_PLAN:
    table_name = csv_file.replace(".csv", "")
    csv_path = os.path.join(".", csv_file)
    
    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_file}")
        continue
        
    # Read sample to infer column names and types
    df_sample = pd.read_csv(csv_path, nrows=100)
    
    # Generate CREATE TABLE statement dynamically
    col_defs = []
    for col in df_sample.columns:
        col_type = get_sql_type(df_sample[col].dtype, col.lower())
        col_defs.append(f'"{col}" {col_type}')
    
    create_table_sql = f'CREATE TABLE {SCHEMA_NAME}."{table_name}" ({", ".join(col_defs)});'
    
    with conn.cursor() as c:
        c.execute(create_table_sql)
        conn.commit()
        
        # COPY data from CSV into newly created table
        with open(csv_path, "r", encoding="utf-8") as f:
            copy_sql = f'COPY {SCHEMA_NAME}."{table_name}" FROM STDIN WITH (FORMAT csv, HEADER true, NULL \'\')'
            c.copy_expert(copy_sql, f)
        conn.commit()
        
        # Verify row count
        c.execute(f'SELECT COUNT(*) FROM {SCHEMA_NAME}."{table_name}";')
        row_count = c.fetchone()[0]
        print(f"✅ Loaded {table_name:<25} | DB Rows: {row_count:,}")

cur.close()
conn.close()
print("\n🎉 ALL CSVs SUCCESSFULLY INGESTED INTO POSTGRESQL!")

Step 1: Connecting to PostgreSQL...
Step 2: Resetting schema 'insight360'...

Step 3: Auto-generating tables from CSV headers & loading data...
✅ Loaded dim_date                  | DB Rows: 730
✅ Loaded dim_store                 | DB Rows: 216
✅ Loaded dim_product               | DB Rows: 4,200
✅ Loaded dim_customer              | DB Rows: 850,000
✅ Loaded fact_sales                | DB Rows: 4,200,000
✅ Loaded fact_returns              | DB Rows: 220,000
✅ Loaded fact_inventory_snapshot   | DB Rows: 968,188
✅ Loaded fact_staffing             | DB Rows: 11,128

🎉 ALL CSVs SUCCESSFULLY INGESTED INTO POSTGRESQL!


In [11]:
import psycopg2

DB_CONFIG = {
    "host": "127.0.0.1",
    "port": "5432",
    "dbname": "insight360_db",
    "user": "postgres",
    "password": "madhu",  # <-- Update password
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

cur.execute("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_schema = 'insight360' AND table_name = 'fact_sales';
""")

print("Actual columns in insight360.fact_sales:")
for row in cur.fetchall():
    print(f" - {row[0]}")

cur.close()
conn.close()

Actual columns in insight360.fact_sales:
 - sales_id
 - date_key
 - store_key
 - product_key
 - customer_key
 - channel
 - quantity
 - unit_price
 - discount_amount
 - net_sales_amount
 - cost_amount
 - loyalty_flag
 - promotion_id
 - return_flag


In [15]:
import psycopg2

DB_CONFIG = {
    "host": "127.0.0.1",
    "port": "5432",
    "dbname": "insight360_db",
    "user": "postgres",
    "password": "madhu",  # <-- Put your password here
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

with open("validate_sql.sql", "r", encoding="utf-8") as f:
    sql_script = f.read()

cur.execute(sql_script)
conn.commit()

print("✅ SQL Data Marts and Analytical Validation completed successfully!")
cur.close()
conn.close()

✅ SQL Data Marts and Analytical Validation completed successfully!


In [14]:
import psycopg2

DB_CONFIG = {
    "host": "127.0.0.1",
    "port": "5432",
    "dbname": "insight360_db",
    "user": "postgres",
    "password": "madhu",  # <-- Your PostgreSQL password
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

cur.execute("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_schema = 'insight360' AND table_name = 'dim_date';
""")

print("Actual columns in insight360.dim_date:")
for row in cur.fetchall():
    print(f" - {row[0]}")

cur.close()
conn.close()

Actual columns in insight360.dim_date:
 - date_key
 - fiscal_year
 - fiscal_quarter
 - fiscal_month_number
 - month_name
 - week_of_year
 - day_of_week
 - is_weekend
 - is_festive_period
 - festive_period_name
 - is_prior_year_baseline


In [4]:
import os
import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

POSTGRES_USER = "postgres"
POSTGRES_PASSWORD = "madhu"
HOST = "127.0.0.1"
PORT = "5432"
DB_NAME = "insight360_db"
SCHEMA_NAME = "insight360"

conn = psycopg2.connect(
    dbname=DB_NAME, user=POSTGRES_USER, password=POSTGRES_PASSWORD, host=HOST, port=PORT
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

# Clean schema reset
cur.execute(f"DROP SCHEMA IF EXISTS {SCHEMA_NAME} CASCADE;")
cur.execute(f"CREATE SCHEMA {SCHEMA_NAME};")

LOAD_PLAN = [
    "dim_date.csv", "dim_store.csv", "dim_product.csv", "dim_customer.csv",
    "fact_sales.csv", "fact_returns.csv", "fact_inventory_snapshot.csv", "fact_staffing.csv"
]

def map_dtype_to_sql(col_name, series):
    col = col_name.lower()
    
    # 1. Alphanumeric / IDs / Keys / Codes / Names are always TEXT
    if any(k in col for k in [
        "id", "key", "code", "zip", "name", "quarter", "tier", "segment", 
        "status", "channel", "method", "email", "city", "state", "country", 
        "brand", "category", "sku", "reason", "promotion", "promo", "flag"
    ]):
        return "TEXT"
    
    # 2. Date columns
    if "date" in col or "timestamp" in col:
        return "DATE"
        
    # 3. Numeric measures & quantities
    if any(k in col for k in ["price", "amount", "cost", "rate", "pct", "discount", "sales", "revenue", "margin", "labor"]):
        return "NUMERIC"
        
    dtype_str = str(series.dtype)
    if "int" in dtype_str:
        return "BIGINT"
    if "float" in dtype_str:
        return "NUMERIC"
    if "bool" in dtype_str:
        return "BOOLEAN"
    
    return "TEXT"

print("Creating tables & loading data...")
for csv_file in LOAD_PLAN:
    table_name = csv_file.replace(".csv", "")
    if os.path.exists(csv_file):
        df_sample = pd.read_csv(csv_file, nrows=100)
        
        col_defs = []
        for col in df_sample.columns:
            sql_type = map_dtype_to_sql(col, df_sample[col])
            col_defs.append(f'"{col}" {sql_type}')

        cur.execute(f'CREATE TABLE {SCHEMA_NAME}."{table_name}" ({", ".join(col_defs)});')
        
        with open(csv_file, "r", encoding="utf-8") as f:
            cur.copy_expert(f'COPY {SCHEMA_NAME}."{table_name}" FROM STDIN WITH (FORMAT csv, HEADER true, NULL \'\')', f)
        
        cur.execute(f'SELECT COUNT(*) FROM {SCHEMA_NAME}."{table_name}";')
        count = cur.fetchone()[0]
        print(f"✅ Loaded {table_name:<28} | DB Rows: {count:,}")
    else:
        print(f"❌ File not found: {csv_file}")

cur.close()
conn.close()
print("\n🎉 ALL CSVs LOADED PERFECTLY!")

Creating tables & loading data...
✅ Loaded dim_date                     | DB Rows: 730
✅ Loaded dim_store                    | DB Rows: 216
✅ Loaded dim_product                  | DB Rows: 4,200
✅ Loaded dim_customer                 | DB Rows: 850,000
✅ Loaded fact_sales                   | DB Rows: 4,200,000
✅ Loaded fact_returns                 | DB Rows: 220,000
✅ Loaded fact_inventory_snapshot      | DB Rows: 968,188
✅ Loaded fact_staffing                | DB Rows: 11,128

🎉 ALL CSVs LOADED PERFECTLY!


In [2]:
echo "__pycache__/" >> .gitignore
echo "*.csv" >> .gitignore
echo "*.pbix" >> .gitignore

SyntaxError: invalid syntax (1282881991.py, line 1)